Evan Edelstein
EN.605.645.82.SP26

# Module 9 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

In [67]:
import json
import random
from copy import deepcopy
from typing import Dict, List, NamedTuple, Tuple, Callable, Set

NBC = NamedTuple("NBC", [("prior", Dict[str, float]), ("probabilities", Dict[str, Dict[str, Dict[str, float]]])])

## Naive Bayes Classifier

For this assignment you will be implementing and evaluating a Naive Bayes Classifier with the same data from last week:

http://archive.ics.uci.edu/ml/datasets/Mushroom

(You should have downloaded it).

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Important</strong>
    <p>
        No Pandas. The only acceptable libraries in this class are those contained in the `environment.yml`. No OOP, either. You can use Dicts, NamedTuples, Data Classes, etc. as your abstract data type (ADT).
    </p>
</div>


You'll first need to calculate all of the necessary probabilities using a `train` function. A flag will control whether or not you use "+1 Smoothing" or not. You'll then need to have a `classify` function that takes your probabilities, a List of instances (possibly a list of 1) and returns a List of Tuples. Each Tuple has the best class in the first position and a dict with a key for every possible class label and the associated *normalized* probability. For example, if we have given the `classify` function a list of 2 observations, we would get the following back:

```
[("e", {"e": 0.98, "p": 0.02}), ("p", {"e": 0.34, "p": 0.66})]
```

when calculating the error rate of your classifier, you should pick the class label with the highest probability; you can write a simple function that takes the Dict and returns that class label.

As a reminder, the Naive Bayes Classifier generates the *unnormalized* probabilities from the numerator of Bayes Rule:

$$P(C|A) \propto P(A|C)P(C)$$

where C is the class and A are the attributes (data). Since the normalizer of Bayes Rule is the *sum* of all possible numerators and you have to calculate them all, the normalizer is just the sum of the probabilities.

You will have the same basic functions as the last module's assignment and some of them can be reused or at least repurposed.

`train` takes training_data and returns a Naive Bayes Classifier (NBC) as a data structure. There are many options including namedtuples and just plain old nested dictionaries. **No OOP**.

```
def train(training_data, smoothing=True):
   # returns the "classifier" (however you decided to represent the probability tables).
```

The `smoothing` value defaults to True. You should handle both cases.

`classify` takes a NBC produced from the function above and applies it to labeled data (like the test set) or unlabeled data (like some new data). (This is not the same `classify` as the pseudocode which classifies only one instance at a time; it can call it though).

```
def classify(nbc, observations, labeled=True):
    # returns a list of tuples, the argmax and the raw data as per the pseudocode.
```

`evaluate` takes a data set with labels (like the training set or test set) and the classification result and calculates the classification error rate:

$$error\_rate=\frac{errors}{n}$$

Do not use anything else as evaluation metric or the submission will be deemed incomplete, ie, an "F". (Hint: accuracy rate is not the error rate!).

`cross_validate` takes the data and uses 10 fold cross validation (from Module 3!) to `train`, `classify`, and `evaluate`. **Remember to shuffle your data before you create your folds**. I leave the exact signature of `cross_validate` to you but you should write it so that you can use it with *any* `classify` function of the same form (using higher order functions and partial application). If you did so last time, you can reuse it for this assignment.

Following Module 3's material (course notes), `cross_validate` should print out a table in exactly the same format. What you are looking for here is a consistent evaluation metric cross the folds. Print the error rate to 4 decimal places. **Do not convert to a percentage.**


To summarize...

Apply the Naive Bayes Classifier algorithm to the Mushroom data set using 10 fold cross validation and the error rate as the evaluation metric. You will do this *twice*. Once with smoothing=True and once with smoothing=False. You should follow up with a brief hypothesis/explanation for the similarities or differences in the results. You may also compare the results to the Decision Tree and why you think they're different (if they are).

### Provided Functions

You do not need to document these.

You can use this function to read the data file.

In [68]:
def parse_data(file_name: str) -> list[list]:
    data = []
    file = open(file_name, "r")
    for line in file:
        datum = line.rstrip().split(",")
        data.append(datum)
    random.shuffle(data)
    return data

You can use this function to create 10 folds for 5x2 cross validation.

In [69]:
def create_folds(xs: list, n: int) -> list[list[list]]:
    k, m = divmod(len(xs), n)
    # be careful of generators...
    return list(xs[i * k + min(i, m) : (i + 1) * k + min(i + 1, m)] for i in range(n))

Put your code after this line:

-----

# I/O and Data Parsing

<a id="parse_attributes"></a>
## parse_attributes

*`parse_attributes` parse an attributes json file given by filename. The file contains a mapping of each feature to a nested map of encoding of the attribute in the data, to the full name of the attribute to be displayed in the tree. Two dictionaries are returned. The first is a map of each feature to a tuple containing the index of that feature in the dataset and the list of attributes in the domain of the feature. The second maps each feature and encoded attribute to the full name of the attribute. Note, the label and its domain should be included in the json file. The order of each feature in the json should match the order they appear in each row of the data set.*

* **filename** str - filepath to a json of features and attributes - The order of each feature should match the order they appear in the data set.


**returns** Tuple[Dict[str, Tuple[int, List[str]]], Dict[str, Dict[str. str]]] - a map of each feature (and label) to a tuple with its position in the dataset and a list of attributes in the domain of the feature, and a nested map of each feature and encoded attribute to the full name of the attribute

In [70]:
def parse_attributes(filename: str) -> Tuple[Dict[str, Tuple[int, List[str]]], Dict[str, Dict[str, str]]]:
    abrv2fullname: Dict[str, Dict[str, str]] = {}
    attributes: Dict[str, Tuple[int, List[str]]] = {}

    with open(filename, "r") as fh:
        data: Dict[str, Dict[str, str]] = json.load(fh)

    for idx, (feature, attrs) in enumerate(data.items()):
        abrv2fullname[feature] = {}
        attributes[feature] = (idx, [])
        for name, code in attrs.items():
            attributes[feature][1].append(name)
            abrv2fullname[feature][code] = name

    return attributes, abrv2fullname

In [71]:
filename = "./agaricus-lepiota-3.attrs.json"
attributes, abrv2fullname = parse_attributes(filename)

attribute_keys = [
    "mushroom-type",
    "cap-shape",
    "cap-surface",
    "cap-color",
    "bruises?",
    "odor",
    "gill-attachment",
    "gill-spacing",
    "gill-size",
    "gill-color",
    "stalk-shape",
    "stalk-root",
    "stalk-surface-above-ring",
    "stalk-surface-below-ring",
    "stalk-color-above-ring",
    "stalk-color-below-ring",
    "veil-type",
    "veil-color",
    "ring-number",
    "ring-type",
    "spore-print-color",
    "population",
    "habitat",
]

assert list(attributes.keys()) == attribute_keys  # test 1 - all keys are present
assert all([len(a) > 1 for a in attr] for _, attr in attributes.values())  # test 2 - all attribute names are full name
assert all([len(k) == 1 and len(v) > 0 for k, v in a.items()] for a in abrv2fullname.values())  # test 3 - can map from single letter to full name

<a id="rename_data"></a>
## rename_data

*`rename_data` Convert all values in data from an encoded attribute to the full attribute name given by the nested dictionary abrv2name, which maps each feature to a dictionary of encoded attribute values to full attribute name. A dictionary that maps each feature to its domain is also required. If a value in data cannot be translated, None is returned.*

* **data** List[List[str]] - a 2d list of observed attributes.
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (and label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **abrv2name** Dict[str, Dict[str. str]] - a nested map of each feature and encoded attribute to the full name of the attribute


**returns** List[List[str]] | None - a copy of data with all values translated to their full name or None if a value cannot be translated

In [72]:
def rename_data(data: List[List[str]], attributes: Dict[str, Tuple[int, List[str]]], abrv2name: Dict[str, Dict[str, str]]) -> List[List[str]] | None:
    new_data = []
    for row in data:
        if len(row) != len(attributes):
            return None

        new_row = []
        for value, attr in zip(row, attributes):
            if attr in abrv2name and value in abrv2name[attr]:
                new_row.append(abrv2name[attr][value])
            else:
                return None
        new_data.append(new_row)

    return new_data

In [73]:
data = [["a", "b", "c"], ["a", "b", "c"]]
attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
abrv2name = {"1": {"a": "aaa"}, "2": {"b": "bbb"}, "3": {"c": "ccc"}}

result = rename_data(data, attributes, abrv2name)
assert result is not None and result[0][0] == "aaa" and result[0][1] == "bbb" and result[0][2] == "ccc" and result[1][0] == "aaa" and result[1][1] == "bbb" and result[1][2] == "ccc"  # test 1 - normal replacement


data = [["a", "b", "c"]]
attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
abrv2name = {"2": {"b": "bbb"}, "3": {"c": "ccc"}}

result = rename_data(data, attributes, abrv2name)
assert result is None  # test 2 - missing attribute in map


data = [["a", "b", "c", "d"]]
attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
abrv2name = {"1": {"a": "aaa"}, "2": {"b": "bbb"}, "3": {"c": "ccc"}}
result = rename_data(data, attributes, abrv2name)
assert result is None  # test 3 - extra value in data

<a id="split_features"></a>
## split_features

*`split_features` Given a mapping where each key is a feature, extract all the keys except the one matching label.* **Used by**: [run_model](#run_model)

* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - key to skip when scanning attributes


**returns** Set[str] - a set of all non-label features 

In [74]:
def split_features(attributes: Dict[str, Tuple[int, List[str]]], label: str) -> List[str]:
    return [i for i in attributes if i != label]

In [75]:
attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
label = "3"
features = split_features(attributes, label)
assert features == ["1", "2"]  # test 1 - splits features and labels

attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
label = "4"
features = split_features(attributes, label)
assert features == ["1", "2", "3"]  # test 2 - label doesn't exist in attributes


attributes = {"1": (0, ["a"])}
label = "1"
features = split_features(attributes, label)
assert features == []  # test 3 - only label so features is empty

# Naive Bayes Classifier

<a id="pretty_print_nbc"></a>
## pretty_print_nbc

*`pretty_print_nbc` pretty print a Naive Bayes Classifier probabilities table.* **Used by**: [run_model](#run_model)

* **model** NBC: named tuple holding the Naive Bayes Classifier. the first entry is the prior probabilities and the second is a nested dictionary mapping label -> feature -> attribute -> probability.


**returns** 

In [ ]:
def pretty_print_nbc(model: NBC):
    attribute_label_probs = {}
    for _, feature_prob in model.probabilities.items():
        for feature, attr_prob in feature_prob.items():
            if feature not in attribute_label_probs:
                attribute_label_probs[feature] = {}
            for attr, p in attr_prob.items():
                if attr in attribute_label_probs[feature]:
                    attribute_label_probs[feature][attr].append(p)
                else:
                    attribute_label_probs[feature][attr] = [p]

    l_str = ",".join([l + f"({round(f, 4)})" for l, f in model.prior.items()])
    print(f"feature,attribute,{l_str}")
    for feature, attr_prob in attribute_label_probs.items():
        for attr, p_list in attr_prob.items():
            p_str = ",".join([str(round(i, 4)) for i in p_list])
            print(f"{feature},{attr},{p_str}")


<a id="subset_data"></a>
## subset_data

*`subset_data` Given a 2d list of data, a column index and a value, return all rows in the input data that have the value at the column index.* **Used by**: [naive_bayes_classifier](#naive_bayes_classifier)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **column_idx** int - position in row to check for match 
* **value** str - value to filter data by
* **shallow** bool - perform shallow copy of each row


**returns** List[List[str]] - rows in data that have value at column index position

In [77]:
def subset_data(data: List[List[str]], column_idx: int, value: str, shallow: bool = False) -> List[List[str]]:
    if shallow:
        return [row for row in data if row[column_idx] == value]
    return [deepcopy(row) for row in data if row[column_idx] == value]

In [78]:
data = [["a", "y"], ["b", "y"], ["a", "n"]]
assert subset_data(data, 0, "a") == [["a", "y"], ["a", "n"]]  # test 1 - get rows

result = subset_data(data, 0, "a")
result[0][0] = "A"
assert data[0][0] == "a"  # test 2.a - deep copy

result = subset_data(data, 0, "a", True)
result[0][0] = "A"
assert data[0][0] == "A"  # test 2.b - shallow copy

assert subset_data(data, 0, "c") == []  # test 3 - empty list if no match

<a id="naive_bayes_classifier"></a>
## naive_bayes_classifier

*`naive_bayes_classifier` calculate label and attribute probabilities from a dataset and store them in a naive bayes classifier model. The model is a named tuple with two entries. The first, model.priors, contains the frequency of each label in the entire dataset. The second, model.probabilities, is a nested dictionary mapping each label, feature and attribute to the number of rows with that feature in label divided by the number of rows with that label. If smoothing is enabled, one is added to the numerator and denominator of attribute probability. This ensures any missing attribute/label pairs in the dataset are still added to the model.* **Uses** [subset_data](#subset_data)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **features** List[str] - list of features
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - label name
* **smoothing** bool - if true apply smoothing
* **trace** bool - if True print debug information

**returns** NBC - naive bayes classifier model

In [79]:
def naive_bayes_classifier(data: List[List[str]], features: List[str], attributes: Dict[str, Tuple[int, List[str]]], label: str, smoothing: bool = True, trace: bool = False) -> NBC:
    smooth_factor = 1 if smoothing else 0
    probabilities: Dict[str, Dict[str, Dict[str, float]]] = {}
    priors: Dict[str, float] = {}
    label_idx, labels = attributes[label]

    for label_value in labels:
        label_rows = subset_data(data, label_idx, label_value)
        label_rows_count = len(label_rows)
        priors[label_value] = label_rows_count / len(data)  # freq of label in dataset
        probabilities[label_value] = {}
        for feature in features:
            feature_idx, domain = attributes[feature]
            probabilities[label_value][feature] = {}
            for attr in domain:
                attribute_row_count = len(subset_data(label_rows, feature_idx, attr))
                probabilities[label_value][feature][attr] = (attribute_row_count + smooth_factor) / (label_rows_count + smooth_factor)  # freq of attribute in label rows

    return NBC(priors, probabilities)

In [ ]:
# TODO

# Model

<a id="train"></a>
## train

*`train` train a naive bayes classifier on a given dataset.* **Uses** [naive_bayes_classifier](#naive-bayes-classifier)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **features** List[str] - set of features
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - label name
* **smoothing** bool - if true apply smoothing
* **trace** bool - if True print debug information

**returns** NBC - naive bayes classifier

In [81]:
def train(data: List[List[str]], features: List[str], attributes: Dict[str, Tuple[int, List[str]]], label: str, smoothing: bool = True, trace=False) -> NBC:
    return naive_bayes_classifier(data, features, attributes, label, smoothing, trace)

In [ ]:
# TODO

<a id="calculate_estimates"></a>
## calculate_estimates

*`calculate_estimates` generate the normalized estimate of each label given a row of observed attributes and a naive bayes classifier. A tuple containing the highest probability label and a dictionary mapping each label to its normalized probability is returned.* **Uses** [naive_bayes_classifier](#naive-bayes-classifier)

* **model** NBC - a named tuple holding the Naive Bayes Classifier. the first entry is the prior probabilities and the second is a nested dictionary mapping label -> feature -> attribute -> probability
* **observations** List[str] - a list of observed attribute
* **features** List[str] - list of features
* **labels** List[str] - list of class labels

**returns** Tuple[str, Dict[str, float]] - the estimated class label and a map of each label to its normalized probability

In [ ]:
def calculate_estimates(model: NBC, observations: List[str], features: List[str], labels: List[str]) -> Tuple[str, Dict[str, float]]:
    estimates: Dict[str, float] = {}
    total_probability = 0
    for label in labels:  # c = p(c) * PI(p(fi|c)) for fi in features for c in labels
        probability = model.prior[label]
        for feature, attr in zip(features, observations):
            probability *= model.probabilities[label][feature][attr]

        estimates[label] = probability
        total_probability += probability

    # normalize
    for label in labels:
        prob = estimates[label]
        estimates[label] = prob / total_probability if total_probability != 0.0 else 0.0

    estimated_label = max(estimates.items(), key=lambda x: x[1])[0]  # label with highest normalized prob
    return estimated_label, estimates

In [ ]:
# TODO

<a id="classify"></a>
## classify

*`classify` Given a naive bayes classifier and a 2d list of observed attributes, estimate the label of each observation.* **Uses** [calculate_estimates](#calculate_estimates)

* **model** NBC - a named tuple holding the Naive Bayes Classifier. the first entry is the prior probabilities and the second is a nested dictionary mapping label -> feature -> attribute -> probability
* **observations** List[List[str]] - a 2d list of observed attributes
* **features** List[str] - list of features
* **labels** List[str] - list of class labels


**returns** List[str] - list of estimated labels for each row in observations

In [85]:
def classify(model: NBC, observations: List[List[str]], features: List[str], labels: List[str]) -> List[Tuple[str, Dict[str, float]]]:
    classifications = []
    for row in observations:
        estimate_label, estimates = calculate_estimates(model, row, features, labels)  # classify from pseudocode
        classifications.append((estimate_label, estimates))
    return classifications

In [ ]:
# TODO

<a id="evaluate"></a>
## evaluate

*`evaluate` Given a list of true labels and a list of estimated labels, count the number of errors and create a confusion matrix by calculating the number TP, TN, FP, FN.*

* **truth_set** List[str] - list of true labels
* **classifications**  List[Tuple[str, Dict[str, float]]] - list of estimated labels


**returns** Tuple[int,Dict[str, int]] - number of non-matching estimates, confusion matrix

In [87]:
def evaluate(truth_set: List[str], classifications: List[Tuple[str, Dict[str, float]]], labels: List[str]) -> Tuple[int, Dict[str, int]]:
    errors = 0
    fold_cm: Dict[str, int] = {"TN": 0, "TP": 0, "FN": 0, "FP": 0}
    for true_label, estimate_prob in zip(truth_set, classifications):
        estimate = estimate_prob[0]  # highest probability label
        if true_label == estimate:
            if estimate == labels[1]:
                fold_cm["TP"] += 1
            else:
                fold_cm["TN"] += 1

        elif true_label != estimate:
            errors += 1
            if true_label == labels[0]:
                fold_cm["FP"] += 1
            else:
                fold_cm["FN"] += 1
    return errors, fold_cm

In [ ]:
# TODO

<a id="divide_folds"></a>
## divide_folds

*`divide_folds` Shuffle a dataset and divide into n leave-one-out training and test pairs.* **Uses** [create_folds](#create_folds) 

* **data** List[List[str]] - 2d list of observations and label
* **n_folds** int - number of folds to generate

**returns** List[Tuple[List[List[str]], List[List[str]]]] - List of tuple containing a training set and test set for each fold

In [89]:
def divide_folds(data: List[List[str]], n_folds: int = 10) -> List[Tuple[List[List[str]], List[List[str]]]]:
    random.shuffle(data)
    folds = create_folds(data, n_folds)

    k_folds = []
    for idx, test_fold in enumerate(folds):
        training_set = []
        for idx2, train_fold in enumerate(folds):
            if idx == idx2:
                continue
            training_set.extend(train_fold)

        k_folds.append((training_set, test_fold))
    return k_folds

In [ ]:
# TODO

<a id="pretty_print_fold"></a>
## pretty_print_fold

*`pretty_print_fold` pretty print the error rate and confusion matrix from a single fold of cross validation.*

* **k** int - fold number
* **training_set** List[List[str]] - 2d list of observations and label to train model on
* **test_set** List[List[str]] - 2d list of observations to test model on
* **error_rate** float - error rate of the fold
* **confusion_matrix** Dict[str, int] - confusion matrix of the fold, represented as a dictionary with keys for TP,TN,FP,FN. 


**returns** 

In [ ]:
def pretty_print_fold(k: int, training_set: List[List[str]], test_set: List[List[str]], error_rate: float, confusion_matrix: Dict[str, int]):
    print(f"Fold {k}")
    print(f"Training size: {len(training_set)} | Test size: {len(test_set)}")
    print(f"Error rate: {error_rate:0.4f}")
    print(f"Confusion matrix:")
    print(f"TP={confusion_matrix['TP']}  FP={confusion_matrix['FP']}\nFN={confusion_matrix['FN']}  TN={confusion_matrix['TN']}")
    print()
    return

In [ ]:
# TODO

<a id="run_fold"></a>
## run_fold

*`run_fold` run a single fold of cross validation by training the model on a training set, classifying it against a test set and evaluating the classification.* **Uses** [pretty_print_fold](#pretty_print_fold)

* **k** int - fold number
* **training_set** List[List[str]] - 2d list of observations and label to train model on
* **test_set** List[List[str]] - 2d list of observations to test model on
* **features** List[str] - list of features
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - key of label in attributes
* **smoothing** bool - if true apply smoothing
* **train_fn** Callable - function to produce model from training data
* **classify_fn** Callable - function to generate label estimates on test data using a model
* **evaluate_fn** Callable - function to collect the number of errors from the classification, as well as, update a confusion matrix.
* **trace** bool - if True print debug information


**returns** Tuple[float, Dict[str, int]] -  error rate and confusion matrix from fold

In [ ]:
def run_fold(
    k: int,
    training_set: List[List[str]],
    test_set: List[List[str]],
    features: List[str],
    attributes: Dict[str, Tuple[int, List[str]]],
    label: str,
    smoothing: bool = True,
    train_fn: Callable = train,
    classify_fn: Callable = classify,
    eval_fn: Callable = evaluate,
    trace: bool = False,
) -> Tuple[float, Dict[str, int]]:
    label_idx, label_values = attributes[label]
    # train
    model = train_fn(training_set, features, attributes, label, smoothing, trace)

    # classify
    masked_test_set = [[i for c, i in enumerate(row) if c != label_idx] for row in test_set]
    classifications = classify_fn(model, masked_test_set, features, label_values)

    # evaluate
    truth_set = [row[label_idx] for row in test_set]
    errors, confusion_matrix = eval_fn(truth_set, classifications, label_values)

    error_rate = errors / len(test_set)
    pretty_print_fold(k, training_set, test_set, error_rate, confusion_matrix)
    return error_rate, confusion_matrix

In [ ]:
# TODO

<a id="cross_validate"></a>
## cross_validate

*`cross_validate` perform n_fold cross validation. For each fold, a train_fn is used to produce a model from the training data in the fold. The model is used to classify the test data in the fold using classify_fn. The classification is evaluated using the evaluate_fn. The error rate and a confusion matrix built from all the folds is returned.* **Uses** [divide_folds](#divide_folds) and [run_fold](#run_fold)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **features** List[str] - set of features
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - label name
* **smoothing** bool - if true apply smoothing
* **train_fn** Callable - function to produce model from training data
* **classify_fn** Callable - function to generate label estimates on test data using a model
* **evaluate_fn** Callable - function to collect the number of errors from the classification, as well as, update a confusion matrix.
* **n_folds** int - number of folds to use
* **trace** bool - if True print debug information

**returns** Tuple[float, Dict[str, int]] -  error rate and confusion matrix from all folds

In [95]:
def cross_validate(
    data: List[List[str]], features: List[str], attributes: Dict[str, Tuple[int, List[str]]], label: str, smoothing: bool = True, train_fn: Callable = train, classify_fn: Callable = classify, eval_fn: Callable = evaluate, n_folds: int = 10, trace: bool = False
) -> Tuple[float, Dict[str, int]]:
    confusion_matrices: List[Dict[str, int]] = []
    total_errors = []

    for k, (training_set, test_set) in enumerate(divide_folds(data, n_folds)):
        error_rate, cm = run_fold(k, training_set, test_set, features, attributes, label, smoothing, train_fn, classify_fn, eval_fn, trace)
        total_errors.append(error_rate)
        confusion_matrices.append(cm)

    total_error_rate = sum(total_errors) / len(total_errors)
    total_cm = {k: sum([cm[k] for cm in confusion_matrices]) for k in confusion_matrices[0]}  # sum cm from each fold
    return total_error_rate, total_cm


In [ ]:
# TODO

# Run

<a id="run_model"></a>
## run_model

*`run_model` perform 10-fold cross validation on a dataset and then print the naive bayes classifier trained on the entire dataset.* **Uses** [split_features](#split_features), [cross_validate](#cross_validate), [train](#train) and [pretty_print_nbc](#pretty_print_nbc)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **features** Set[str] - set of features
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - label name
* **smoothing** bool - if true apply smoothing
* **trace** bool - if True print debug information

**returns** 

In [97]:
def run_model(data: List[List[str]], attributes: Dict[str, Tuple[int, List[str]]], label: str, smoothing: bool = True, trace: bool = False):
    n_folds = 10

    features = split_features(attributes, label)

    avrg_error_rate, cm = cross_validate(data, features, attributes, label, smoothing=smoothing, n_folds=n_folds)

    print(f"\nTotal Confusion Matrix ({n_folds}-fold CV):")
    print(f"TP={cm['TP']}  FP={cm['FP']}\nFN={cm['FN']}  TN={cm['TN']}")
    print(f"\nAverage Error Rate ({n_folds}-fold CV): {avrg_error_rate:.4f}")

    model = train(data, features, attributes, label, smoothing, trace=trace)
    assert model is not None

    print()
    print("NBC model:")
    pretty_print_nbc(model)
    print()

In [98]:
attributes = {"Shape": (0, ["round", "square"]), "Size": (1, ["large", "small"]), "Color": (2, ["blue", "green", "red"]), "Safe?": (3, ["yes", "no"])}
label = "Safe?"

data = [
    ["round", "large", "blue", "no"],
    ["square", "large", "green", "yes"],
    ["square", "small", "red", "no"],
    ["round", "large", "red", "yes"],
    ["square", "small", "blue", "no"],
    ["round", "small", "blue", "no"],
    ["round", "small", "red", "yes"],
    ["square", "small", "green", "no"],
    ["round", "large", "green", "yes"],
    ["square", "large", "green", "yes"],
    ["square", "large", "red", "no"],
    ["square", "large", "green", "yes"],
    ["round", "large", "red", "yes"],
    ["square", "small", "red", "no"],
    ["round", "small", "green", "no"],
]
print("Smoothing on")
run_model(data, attributes, label)

print("Smoothing off")
run_model(data, attributes, label, smoothing=False)

Smoothing on
Fold 0
Training size: 13 | Test size: 2
Error rate: 0.0000
Confusion matrix:
TP=1  FP=0
FN=0  TN=1

Fold 1
Training size: 13 | Test size: 2
Error rate: 0.5000
Confusion matrix:
TP=1  FP=0
FN=1  TN=0

Fold 2
Training size: 13 | Test size: 2
Error rate: 0.5000
Confusion matrix:
TP=0  FP=0
FN=1  TN=1

Fold 3
Training size: 13 | Test size: 2
Error rate: 0.0000
Confusion matrix:
TP=1  FP=0
FN=0  TN=1

Fold 4
Training size: 13 | Test size: 2
Error rate: 0.0000
Confusion matrix:
TP=0  FP=0
FN=0  TN=2

Fold 5
Training size: 14 | Test size: 1
Error rate: 0.0000
Confusion matrix:
TP=0  FP=0
FN=0  TN=1

Fold 6
Training size: 14 | Test size: 1
Error rate: 0.0000
Confusion matrix:
TP=1  FP=0
FN=0  TN=0

Fold 7
Training size: 14 | Test size: 1
Error rate: 1.0000
Confusion matrix:
TP=0  FP=1
FN=0  TN=0

Fold 8
Training size: 14 | Test size: 1
Error rate: 0.0000
Confusion matrix:
TP=1  FP=0
FN=0  TN=0

Fold 9
Training size: 14 | Test size: 1
Error rate: 1.0000
Confusion matrix:
TP=0  FP=0

In [99]:
trace = False
data = parse_data("./agaricus-lepiota-1-2.data")
attributes, abrv2name = parse_attributes("./agaricus-lepiota-3.attrs.json")
label = "mushroom-type"

data = rename_data(data, attributes, abrv2name)
assert data is not None

print("Smoothing on")
run_model(data, attributes, label)

print("Smoothing off")
run_model(data, attributes, label, smoothing=False)

Smoothing on
Fold 0
Training size: 7311 | Test size: 813
Error rate: 0.0504
Confusion matrix:
TP=418  FP=37
FN=4  TN=354

Fold 1
Training size: 7311 | Test size: 813
Error rate: 0.0418
Confusion matrix:
TP=423  FP=34
FN=0  TN=356

Fold 2
Training size: 7311 | Test size: 813
Error rate: 0.0541
Confusion matrix:
TP=403  FP=44
FN=0  TN=366

Fold 3
Training size: 7311 | Test size: 813
Error rate: 0.0418
Confusion matrix:
TP=430  FP=30
FN=4  TN=349

Fold 4
Training size: 7312 | Test size: 812
Error rate: 0.0530
Confusion matrix:
TP=407  FP=37
FN=6  TN=362

Fold 5
Training size: 7312 | Test size: 812
Error rate: 0.0443
Confusion matrix:
TP=440  FP=34
FN=2  TN=336

Fold 6
Training size: 7312 | Test size: 812
Error rate: 0.0419
Confusion matrix:
TP=418  FP=30
FN=4  TN=360

Fold 7
Training size: 7312 | Test size: 812
Error rate: 0.0419
Confusion matrix:
TP=412  FP=33
FN=1  TN=366

Fold 8
Training size: 7312 | Test size: 812
Error rate: 0.0443
Confusion matrix:
TP=413  FP=34
FN=2  TN=363

Fold 9

# Discussion: 
### Compare smoothing

### Compare NBC to DT

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.